# Pit Stop Submission Blender: 0.95437 Anchor + H-Blend Research

This notebook treats `0.95437` as the new anchor submission. `0.95435` is the closest high-scoring support, while `0.95431` provides a more diverse support profile. Older `0.95419`, `0.95418`, and `0.95411` submissions remain available for diagnostics and controlled blends.

The notebook tests a compact set of methods: micro probability blends, rank blends, HB10-style row-wise h-blends, and selective corrections. Final files are saved to `outputs/max` and `outputs/pro`, with `outputs/report.csv` as the submit guide.


In [ ]:
"""Import libraries, configure paths, and define compact display helpers."""

from pathlib import Path
from html import escape
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
from IPython.display import display, HTML

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("ggplot")

ID_COL = "id"
TARGET_COL = "PitNextLap"
CLIP_LOW = 1e-7
CLIP_HIGH = 1 - 1e-7

DATASET_CANDIDATES = [
    Path("/kaggle/input/pitstop-blend-inputs"),
    Path("/kaggle/input/pit-stop-blend-inputs"),
    Path("/kaggle/input/blend-dataset"),
    Path("/kaggle/input") / "blend_dataset",
    Path("competitions/predicting-pit-stop/blend_dataset"),
    Path("PredictingPitStop/blend_dataset"),
    Path("blend_dataset"),
]

OUTPUT_ROOT = Path("outputs")
MAX_ROOT = OUTPUT_ROOT / "max"
PRO_ROOT = OUTPUT_ROOT / "pro"
DIAGNOSTIC_ROOT = OUTPUT_ROOT / "diagnostics"
REPORT_PATH = OUTPUT_ROOT / "report.csv"

for folder in [MAX_ROOT, PRO_ROOT, DIAGNOSTIC_ROOT]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
if REPORT_PATH.exists():
    REPORT_PATH.unlink()


def find_dataset_dir():
    for path in DATASET_CANDIDATES:
        if (path / "public").exists() and (path / "ours").exists():
            return path
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.rglob("*")):
            if path.is_dir() and (path / "public").exists() and (path / "ours").exists():
                return path
    raise FileNotFoundError("Could not find a dataset folder containing public/ and ours/ subfolders.")


def show_title(title, subtitle=None):
    subtitle_html = f'<div style="color:#6b7280;font-size:13px;margin-top:4px;">{escape(str(subtitle))}</div>' if subtitle else ""
    display(HTML(
        f"""
        <div style="margin:20px 0 12px 0;padding-bottom:9px;border-bottom:1px solid #e5e7eb;">
          <div style="font-size:20px;font-weight:750;color:#111827;">{escape(str(title))}</div>
          {subtitle_html}
        </div>
        """
    ))


def show_cards(metrics, columns=4):
    width = max(1, int(100 / columns))
    cards = []
    for label, value in metrics.items():
        if isinstance(value, float):
            value = f"{value:.6f}".rstrip("0").rstrip(".")
        elif isinstance(value, int):
            value = f"{value:,}"
        cards.append(
            f"""
            <div style="box-sizing:border-box;width:{width}%;padding:6px;">
              <div style="border:1px solid #e5e7eb;border-radius:8px;padding:12px;background:#fff;">
                <div style="font-size:12px;color:#6b7280;text-transform:uppercase;letter-spacing:.03em;">{escape(str(label))}</div>
                <div style="font-size:21px;font-weight:750;color:#111827;margin-top:5px;">{escape(str(value))}</div>
              </div>
            </div>
            """
        )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;margin:0 -6px 14px -6px;">{"".join(cards)}</div>'))


def show_table(title, df, max_rows=12, precision=6):
    shown = df.head(max_rows).copy()
    display(HTML(f'<div style="font-size:16px;font-weight:700;color:#111827;margin:14px 0 6px;">{escape(title)}</div>'))
    styler = shown.style.format(precision=precision).set_table_styles([
        {"selector":"th", "props":[("background", "#f9fafb"), ("color", "#374151"), ("font-weight", "700"), ("border-bottom", "1px solid #e5e7eb")]},
        {"selector":"td", "props":[("border-bottom", "1px solid #f3f4f6"), ("font-size", "13px")]},
        {"selector":"table", "props":[("border-collapse", "collapse"), ("width", "100%")]},
    ])
    display(styler)
    if len(df) > max_rows:
        display(HTML(f'<div style="color:#6b7280;font-size:12px;margin-top:4px;">Showing {max_rows} of {len(df)} rows.</div>'))


def corr(a, b):
    return float(np.corrcoef(np.asarray(a), np.asarray(b))[0, 1])


def mean_abs_delta(a, b):
    return float(np.abs(np.asarray(a) - np.asarray(b)).mean())


def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), CLIP_LOW, CLIP_HIGH)


## 1. Load Blend Dataset

The loader reads the structured `blend_dataset`. The current `public/super` group contains the new `0.95437` anchor, the close `0.95435` support, the `0.95431` support files, and older high-scoring submissions used for controlled diversity.


In [ ]:
"""Load all valid submission files from the structured blend dataset."""

dataset_dir = find_dataset_dir()
predictions = {}
records = []
base_ids = None


def load_submission(path, source):
    global base_ids
    df = pd.read_csv(path)
    if ID_COL not in df.columns:
        return None
    target_candidates = [col for col in df.columns if col != ID_COL]
    if TARGET_COL in df.columns:
        target_col = TARGET_COL
    elif len(target_candidates) == 1:
        target_col = target_candidates[0]
    else:
        return None

    df = df[[ID_COL, target_col]].rename(columns={target_col: TARGET_COL})
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    if df[TARGET_COL].isna().any() or df[ID_COL].duplicated().any():
        return None

    if base_ids is None:
        base_ids = df[ID_COL].copy()
    elif not base_ids.equals(df[ID_COL]):
        return None

    group = path.parent.name
    safe_stem = path.stem.replace(".", "_").replace("-", "_")
    name = f"{source}_{group}_{safe_stem}"
    pred = clip_pred(df[TARGET_COL].to_numpy(dtype=float))
    predictions[name] = pred

    public_score = np.nan
    if source == "public":
        try:
            public_score = float(path.stem.split("_")[0])
        except ValueError:
            public_score = np.nan

    return {
        "name": name,
        "source": source,
        "group": group,
        "public_score": public_score,
        "rows": len(df),
        "mean": pred.mean(),
        "std": pred.std(),
        "file": str(path.relative_to(dataset_dir)),
    }


for source in ["public", "ours"]:
    folder = dataset_dir / source
    if folder.exists():
        for path in sorted(folder.rglob("*.csv")):
            row = load_submission(path, source)
            if row is not None:
                records.append(row)

if not records:
    raise FileNotFoundError("No valid submissions were loaded from the structured dataset.")

input_summary = pd.DataFrame(records).sort_values(
    ["source", "group", "public_score", "name"], ascending=[True, True, False, True]
).reset_index(drop=True)
input_summary.to_csv(DIAGNOSTIC_ROOT / "inputs.csv", index=False)

show_title("Loaded blend dataset", f"Dataset folder: {dataset_dir}")
show_cards({
    "input files": len(input_summary),
    "public files": int(input_summary["source"].eq("public").sum()),
    "own files": int(input_summary["source"].eq("ours").sum()),
    "rows": len(base_ids),
})
show_table(
    "Structured inputs",
    input_summary[["name", "source", "group", "public_score", "mean", "std", "file"]],
    max_rows=28,
)


## 2. Anchor Diagnostics

The new `0.95437` file becomes the anchor. We compare it with `0.95435`, `0.95431`, `0.95419`, and older support files to understand which sources are close enough for conservative blending and which are diverse enough for h-blend experiments.


In [ ]:

"""Select the new anchor and compare the strongest public support signals."""

super_meta = input_summary[input_summary["group"].eq("super")].sort_values("public_score", ascending=False).copy()
if super_meta.empty:
    raise RuntimeError("No public/super submissions were found.")


def super_names_by_score(score):
    return super_meta[np.isclose(super_meta["public_score"], score, atol=1e-8)]["name"].tolist()

s37_names = super_names_by_score(0.95437)
if not s37_names:
    raise RuntimeError("The 0.95437 submission is required in public/super.")
s37_name = s37_names[0]
s37_pred = predictions[s37_name]

s35_names = super_names_by_score(0.95435)
if not s35_names:
    raise RuntimeError("The 0.95435 support submission is required in public/super.")
s35_name = s35_names[0]
s35_pred = predictions[s35_name]

s31_names = super_names_by_score(0.95431)
if not s31_names:
    raise RuntimeError("At least one 0.95431 support submission is required in public/super.")
s31_pred = clip_pred(np.vstack([predictions[name] for name in s31_names]).mean(axis=0))

s19_names = super_names_by_score(0.95419)
s18_names = super_names_by_score(0.95418)
s11_names = super_names_by_score(0.95411)
if not s19_names or not s18_names or not s11_names:
    raise RuntimeError("The older 0.95419, 0.95418, and 0.95411 support files are required.")
s19_pred = predictions[s19_names[0]]
s18_pred = predictions[s18_names[0]]
s11_pred = clip_pred(np.vstack([predictions[name] for name in s11_names]).mean(axis=0))

super_names = super_meta["name"].tolist()
super_matrix = pd.DataFrame({name: predictions[name] for name in super_names})
super_corr = super_matrix.corr()

anchor_rows = []
for name in super_names:
    pred = predictions[name]
    score = float(super_meta.loc[super_meta["name"].eq(name), "public_score"].iloc[0])
    anchor_rows.append({
        "signal": name,
        "public_score": score,
        "mean": pred.mean(),
        "std": pred.std(),
        "corr_to_s37": corr(pred, s37_pred),
        "delta_to_s37": mean_abs_delta(pred, s37_pred),
    })
anchor_summary = pd.DataFrame(anchor_rows).sort_values("public_score", ascending=False).reset_index(drop=True)
anchor_summary.to_csv(DIAGNOSTIC_ROOT / "anchor_diagnostics.csv", index=False)

show_title("Anchor diagnostics", "0.95437 is the new base; 0.95435 is the closest support")
show_cards({
    "anchor": "0.95437",
    "s35 delta": mean_abs_delta(s35_pred, s37_pred),
    "s35 corr": corr(s35_pred, s37_pred),
    "super files": len(super_names),
})
show_table("Super signals", anchor_summary, max_rows=len(anchor_summary))

fig, ax = plt.subplots(figsize=(8, 5.5))
if sns is not None:
    sns.heatmap(super_corr, cmap="viridis", annot=True, fmt=".5f", cbar=False, ax=ax)
else:
    im = ax.imshow(super_corr.values, cmap="viridis")
    ax.set_xticks(range(len(super_corr.columns)), super_corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(super_corr.index)), super_corr.index)
    fig.colorbar(im, ax=ax)
ax.set_title("Super submission correlation")
plt.tight_layout()
plt.show()


## 3. Build Support Signals

This block prepares compact support signals used by the final methods. The old `tb` region, `s35`, and the `s31` mean are not saved directly; they are used as controlled correction sources around `s37`.


In [ ]:

"""Build support signals used by probability, rank, h-blend, and selective methods."""


def names_by_group(group):
    return input_summary[input_summary["group"].eq(group)]["name"].tolist()


def mean_predictions(names):
    if not names:
        raise RuntimeError("Cannot average an empty list of predictions.")
    return clip_pred(np.vstack([predictions[name] for name in names]).mean(axis=0))

public_core = input_summary[input_summary["group"].eq("core")].sort_values("public_score", ascending=False)["name"].head(6).tolist()
public_diverse = names_by_group("diverse")
top_external = names_by_group("top_external")
if not public_core or not public_diverse or not top_external:
    raise RuntimeError("Core, diverse, and top_external groups are required for reference signals.")

core_pred = mean_predictions(public_core)
diverse_pred = predictions[public_diverse[0]]
b10_pred = clip_pred(0.950 * core_pred + 0.050 * diverse_pred)
tx_pred = mean_predictions(top_external)
tb_pred = clip_pred(0.900 * tx_pred + 0.100 * b10_pred)
consensus_pred = clip_pred(0.50 * s35_pred + 0.30 * s31_pred + 0.20 * s19_pred)

support_signals = {
    "s37": s37_pred,
    "s35": s35_pred,
    "s31": s31_pred,
    "s19": s19_pred,
    "s18": s18_pred,
    "s11": s11_pred,
    "tb": tb_pred,
    "consensus": consensus_pred,
}

support_summary = pd.DataFrame([
    {
        "signal": name,
        "mean": pred.mean(),
        "std": pred.std(),
        "corr_to_s37": corr(pred, s37_pred),
        "delta_to_s37": mean_abs_delta(pred, s37_pred),
    }
    for name, pred in support_signals.items()
]).sort_values("delta_to_s37").reset_index(drop=True)
support_summary.to_csv(DIAGNOSTIC_ROOT / "support_signals.csv", index=False)

show_title("Support signals", "s35 is the closest support; s31 provides stronger diversity")
show_table("Support summary", support_summary, max_rows=len(support_summary))


## 4. Method 1: Micro Probability Blends

These candidates test whether a very small probability blend with `s35`, `s31`, or older support can improve the new anchor. Because `s37` is already stronger, all weights are conservative.


In [ ]:

"""Create conservative probability-blend candidates around the 0.95437 anchor."""

prob_candidates = {
    "l35": {
        "pred": 0.980 * s37_pred + 0.020 * s35_pred,
        "formula": "0.980*s37 + 0.020*s35",
        "tier": "max",
        "priority": 2,
        "reason": "Main micro blend with the closest 0.95435 support.",
    },
    "l31": {
        "pred": 0.980 * s37_pred + 0.020 * s31_pred,
        "formula": "0.980*s37 + 0.020*s31",
        "tier": "pro",
        "priority": 6,
        "reason": "Small blend with the more diverse 0.95431 support.",
    },
    "lc": {
        "pred": 0.970 * s37_pred + 0.030 * consensus_pred,
        "formula": "0.970*s37 + 0.030*consensus",
        "tier": "pro",
        "priority": 7,
        "reason": "Small blend with a consensus support signal.",
    },
}

prob_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s37": corr(spec["pred"], s37_pred), "delta_to_s37": mean_abs_delta(spec["pred"], s37_pred), "reason": spec["reason"]}
    for key, spec in prob_candidates.items()
]).sort_values("delta_to_s37")
prob_summary.to_csv(DIAGNOSTIC_ROOT / "probability_candidates.csv", index=False)

show_title("Micro probability blends", "Small global corrections from close and diverse support signals")
show_table("Probability candidates", prob_summary, max_rows=len(prob_summary))


## 5. Method 2: Rank Blends

Rank blends preserve the `s37` probability distribution while changing the order of rows. This directly tests whether the support signal improves AUC-style ranking without over-smoothing probabilities.


In [ ]:

"""Create rank-blend candidates that preserve the anchor distribution."""


def normalized_rank(values):
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.linspace(0.0, 1.0, len(values))
    return ranks


def rank_blend(anchor, support, support_weight):
    anchor_rank = normalized_rank(anchor)
    support_rank = normalized_rank(support)
    blended_rank = (1 - support_weight) * anchor_rank + support_weight * support_rank
    order = np.argsort(blended_rank, kind="mergesort")
    sorted_anchor_values = np.sort(anchor)
    out = np.empty_like(anchor, dtype=float)
    out[order] = sorted_anchor_values
    return clip_pred(out)

rank_candidates = {
    "r35": {
        "pred": rank_blend(s37_pred, s35_pred, 0.020),
        "formula": "rank 0.980*s37 + 0.020*s35",
        "tier": "max",
        "priority": 3,
        "reason": "Main rank test using the closest 0.95435 support.",
    },
    "r31": {
        "pred": rank_blend(s37_pred, s31_pred, 0.020),
        "formula": "rank 0.980*s37 + 0.020*s31",
        "tier": "pro",
        "priority": 8,
        "reason": "Rank test using the more diverse 0.95431 support.",
    },
}

rank_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s37": corr(spec["pred"], s37_pred), "delta_to_s37": mean_abs_delta(spec["pred"], s37_pred), "reason": spec["reason"]}
    for key, spec in rank_candidates.items()
]).sort_values("delta_to_s37")
rank_summary.to_csv(DIAGNOSTIC_ROOT / "rank_candidates.csv", index=False)

show_title("Rank blends", "Ordering corrections with the original s37 probability distribution")
show_table("Rank candidates", rank_summary, max_rows=len(rank_summary))


## 6. Method 3: HB10-Style H-Blend

The HB10 notebook uses row-wise ordering: each submission gets a small weight adjustment depending on whether it is high or low among the submissions for that row. This block adapts that idea with `s37` as the dominant source.


In [ ]:

"""Create HB10-style row-wise h-blend candidates with s37 as the dominant source."""


def h_blend_matrix(named_preds, base_weights, subwts, desc_weight=0.70):
    names = list(named_preds)
    matrix = np.vstack([named_preds[name] for name in names]).T
    base = np.asarray([base_weights[name] for name in names], dtype=float)
    corr_w = np.asarray(subwts, dtype=float)
    if len(corr_w) != len(names):
        raise ValueError("subwts length must match number of h-blend submissions.")

    def side(reverse):
        order = np.argsort(-matrix if reverse else matrix, axis=1)
        out = np.zeros(matrix.shape[0], dtype=float)
        for rank_idx in range(matrix.shape[1]):
            col_idx = order[:, rank_idx]
            weights = base[col_idx] + corr_w[rank_idx]
            out += matrix[np.arange(matrix.shape[0]), col_idx] * weights
        return out

    desc = side(True)
    asc = side(False)
    return clip_pred(desc_weight * desc + (1 - desc_weight) * asc)

h_sources = {"s37": s37_pred, "s35": s35_pred, "s31": s31_pred, "s19": s19_pred}

hb37_pred = h_blend_matrix(
    h_sources,
    base_weights={"s37": 0.60, "s35": 0.18, "s31": 0.14, "s19": 0.08},
    subwts=[-0.035, 0.005, 0.012, 0.018],
    desc_weight=0.70,
)
hc37_pred = clip_pred(0.800 * hb37_pred + 0.200 * s37_pred)
hbold_pred = h_blend_matrix(
    h_sources,
    base_weights={"s37": 0.52, "s35": 0.22, "s31": 0.16, "s19": 0.10},
    subwts=[-0.055, 0.010, 0.018, 0.027],
    desc_weight=0.70,
)

hblend_candidates = {
    "hb37": {"pred": hb37_pred, "formula": "HB10-style h-blend: s37/s35/s31/s19", "tier": "max", "priority": 4, "reason": "Main row-wise h-blend adapted from HB10 with s37 as dominant source."},
    "hc37": {"pred": hc37_pred, "formula": "0.800*hb37 + 0.200*s37", "tier": "pro", "priority": 9, "reason": "Conservative h-blend pulled back toward the anchor."},
    "hbold": {"pred": hbold_pred, "formula": "bolder HB10-style h-blend: s37/s35/s31/s19", "tier": "pro", "priority": 10, "reason": "More aggressive h-blend to test row-wise diversity."},
}

hblend_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s37": corr(spec["pred"], s37_pred), "delta_to_s37": mean_abs_delta(spec["pred"], s37_pred), "reason": spec["reason"]}
    for key, spec in hblend_candidates.items()
]).sort_values("delta_to_s37")
hblend_summary.to_csv(DIAGNOSTIC_ROOT / "hblend_candidates.csv", index=False)

show_title("HB10-style h-blends", "Row-wise rank-aware blending around the new anchor")
show_table("H-blend candidates", hblend_summary, max_rows=len(hblend_summary))


## 7. Method 4: Selective Correction

Selective correction changes only a small group of rows where `s37` and the support signal disagree most. It is a safer way to test corrections when the anchor is already strong.


In [ ]:

"""Create selective correction candidates on the largest anchor/support disagreements."""


def selective_blend(anchor, support, fraction, support_weight, extreme_only=False):
    anchor = np.asarray(anchor, dtype=float)
    support = np.asarray(support, dtype=float)
    delta = support - anchor
    eligible = np.ones_like(delta, dtype=bool)
    if extreme_only:
        ranks = normalized_rank(anchor)
        eligible = eligible & ((ranks <= 0.10) | (ranks >= 0.90))
    scores = np.where(eligible, np.abs(delta), -np.inf)
    n_select = max(1, int(round(len(anchor) * fraction)))
    selected = np.argpartition(scores, -n_select)[-n_select:]
    selected = selected[np.isfinite(scores[selected])]
    out = anchor.copy()
    out[selected] = (1 - support_weight) * anchor[selected] + support_weight * support[selected]
    return clip_pred(out), len(selected)

c35_pred, c35_rows = selective_blend(s37_pred, s35_pred, fraction=0.020, support_weight=0.100)
cc_pred, cc_rows = selective_blend(s37_pred, consensus_pred, fraction=0.020, support_weight=0.100)
ex35_pred, ex35_rows = selective_blend(s37_pred, s35_pred, fraction=0.020, support_weight=0.100, extreme_only=True)

selective_candidates = {
    "c35": {"pred": c35_pred, "formula": "top 2% |s35-s37|: 0.900*s37 + 0.100*s35", "tier": "pro", "priority": 11, "changed_rows": c35_rows, "reason": "Selective correction from the closest support."},
    "cc": {"pred": cc_pred, "formula": "top 2% |consensus-s37|: 0.900*s37 + 0.100*consensus", "tier": "pro", "priority": 12, "changed_rows": cc_rows, "reason": "Selective correction from the broader consensus signal."},
    "ex35": {"pred": ex35_pred, "formula": "extreme top/bottom 10%, top 2% |s35-s37| correction", "tier": "pro", "priority": 13, "changed_rows": ex35_rows, "reason": "Extreme-rank correction for AUC-sensitive rows."},
}

selective_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "changed_rows": spec["changed_rows"], "corr_to_s37": corr(spec["pred"], s37_pred), "delta_to_s37": mean_abs_delta(spec["pred"], s37_pred), "reason": spec["reason"]}
    for key, spec in selective_candidates.items()
]).sort_values("delta_to_s37")
selective_summary.to_csv(DIAGNOSTIC_ROOT / "selective_candidates.csv", index=False)

show_title("Selective corrections", "Small row-level changes around the anchor")
show_table("Selective candidates", selective_summary, max_rows=len(selective_summary))


## 8. Save Final Outputs

The saved set is intentionally small at the top. `max` contains the first submissions to try. `pro` contains diagnostics for the next iteration if one method family shows promise.


In [ ]:

"""Save selected candidates to outputs/max and outputs/pro, then build one report file."""


def save_submission(path, pred):
    pred = clip_pred(pred)
    pd.DataFrame({ID_COL: base_ids.values, TARGET_COL: pred}).to_csv(path, index=False)

all_candidates = {"s37": {"pred": s37_pred, "method": "anchor", "tier": "max", "priority": 1, "formula": "0.95437 raw anchor", "changed_rows": 0, "reason": "Verify the new strongest anchor unchanged."}}
for key, spec in prob_candidates.items():
    all_candidates[key] = {"method": "probability", "changed_rows": len(s37_pred), **spec}
for key, spec in rank_candidates.items():
    all_candidates[key] = {"method": "rank", "changed_rows": len(s37_pred), **spec}
for key, spec in hblend_candidates.items():
    all_candidates[key] = {"method": "hblend", "changed_rows": len(s37_pred), **spec}
for key, spec in selective_candidates.items():
    all_candidates[key] = {"method": "selective", **spec}

rows = []
for key, spec in sorted(all_candidates.items(), key=lambda item: item[1]["priority"]):
    pred = clip_pred(spec["pred"])
    folder = MAX_ROOT if spec["tier"] == "max" else PRO_ROOT
    file_path = folder / f"{key}.csv"
    save_submission(file_path, pred)
    rows.append({"priority": spec["priority"], "tier": spec["tier"], "candidate": key, "method": spec["method"], "file": str(file_path.relative_to(OUTPUT_ROOT)), "formula": spec["formula"], "mean": pred.mean(), "std": pred.std(), "corr_to_s37": corr(pred, s37_pred), "delta_to_s37": mean_abs_delta(pred, s37_pred), "changed_rows": spec["changed_rows"], "reason": spec["reason"]})

report = pd.DataFrame(rows).sort_values("priority").reset_index(drop=True)
report.to_csv(REPORT_PATH, index=False)

show_title("Saved output files", "Upload max first; use pro for method diagnostics")
show_cards({"max files": int(report["tier"].eq("max").sum()), "pro files": int(report["tier"].eq("pro").sum()), "total files": len(report), "report": "outputs/report.csv"})
show_table("Submit order", report[["priority", "tier", "candidate", "method", "file", "formula", "delta_to_s37", "changed_rows", "reason"]], max_rows=len(report))


## 9. Candidate Movement

The plots compare how much each candidate moves from `s37`. If `hb37` improves the leaderboard, the next iteration should tune only the h-blend weights. If `l35` or `r35` improves, the next search should stay around small `s35` weights.


In [ ]:

"""Visualize candidate movement and correction shape relative to the new anchor."""

plot_df = report[report["candidate"].ne("s37")].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
if sns is not None:
    sns.barplot(data=plot_df, y="candidate", x="delta_to_s37", hue="method", dodge=False, ax=axes[0])
    sns.barplot(data=plot_df, y="candidate", x="changed_rows", hue="method", dodge=False, ax=axes[1])
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
else:
    axes[0].barh(plot_df["candidate"], plot_df["delta_to_s37"])
    axes[1].barh(plot_df["candidate"], plot_df["changed_rows"])
axes[0].set_title("Movement from s37")
axes[0].set_xlabel("mean absolute delta")
axes[1].set_title("Rows changed")
axes[1].set_xlabel("rows")
for ax in axes:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.3))
for candidate in ["l35", "r35", "hb37"]:
    row = report[report["candidate"].eq(candidate)].iloc[0]
    pred = pd.read_csv(OUTPUT_ROOT / row["file"])[TARGET_COL].to_numpy(dtype=float)
    ax.hist(pred - s37_pred, bins=80, alpha=0.45, label=candidate)
ax.set_title("Main candidate corrections relative to s37")
ax.set_xlabel("prediction delta")
ax.set_ylabel("rows")
ax.legend()
plt.tight_layout()
plt.show()

show_title("Final recommendation", "The max files test the main hypotheses")
show_table("Max candidates", report[report["tier"].eq("max")][["priority", "candidate", "method", "file", "reason"]], max_rows=10)
show_table("Pro candidates", report[report["tier"].eq("pro")][["priority", "candidate", "method", "file", "reason"]], max_rows=20)


## Final Notes

The first files to try are `s37`, `l35`, `r35`, and `hb37`. They answer different questions: raw anchor strength, probability correction from `0.95435`, rank correction from `0.95435`, and HB10-style row-wise diversity.

If none of the max files improves `0.95437`, the next step should not be a broad random search. It should either tune the best-performing method family or add another genuinely different high-scoring external submission.
